In [52]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    silhouette_score
)

df = pd.read_csv("battery_observations.csv", index_col=0)
meta = pd.read_csv("battery_metadata.csv", index_col=0)
df2 = df.join(meta[["climate_zone", "chemistry"]])
df2 = df2.sort_values(["battery_id", "cycle_index"])
df2 = df2.reset_index()
meta = meta.reset_index()
g = lambda x: int(x.split('-')[-1])
df2["battery_id"] = df2["battery_id"].apply(g)
meta["battery_id"] = meta["battery_id"].apply(g)
battery_ids = list(set(df2["battery_id"]))
materials = list(set(meta["chemistry"]))
climate_zones = list(set(meta["climate_zone"]))
labels = list(set(df2["degradation_status"]))

train_labels = pd.read_csv("train_batteries.csv")["battery_id"].apply(g)
val_labels = pd.read_csv("validation_batteries.csv")["battery_id"].apply(g)

train = df2[df2["battery_id"].isin(train_labels)]
val = df2[df2["battery_id"].isin(val_labels)]

In [53]:

cycle_features = [
    "soc_start_pct",
    "soc_end_pct",
    "depth_of_discharge_pct",
    "high_soc_exposure_fraction",
    "low_soc_exposure_fraction",
    "charge_c_rate",
    "discharge_c_rate",
    "charge_current_a",
    "discharge_current_a",
    "charge_duration_min",
    "discharge_duration_min",
    "energy_discharged_kwh",
    "ambient_temp_c",
    #"pack_temp_c",
    "cooling_active",
    "pack_voltage_mean_v"
]

extended_features = cycle_features + [
    "battery_age_days",
    "time_since_previous_cycle_hours",
    "equivalent_full_cycles",
    "cumulative_high_temp_hours",
    "pack_voltage_charge_v",
    "pack_voltage_discharge_v"
]

features_A = [
    "battery_age_days",
    "equivalent_full_cycles",
]

features_B = features_A + [
    "internal_resistance_mOhm",
]

features_C = features_B + [
    "cumulative_high_temp_hours",
]


In [54]:
class_names = ["Normal", "Moderate", "Severe", "Critical"]
class_names_mapping = {class_names[i]:i for i in range(len(class_names))}

def perform_knn(k, X_train, X_val, y_train, y_val):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    predicted = knn.predict(X_val)

    predicted_num = [class_names_mapping[x] for x in predicted]
    y_val_num = [class_names_mapping[x] for x in y_val]

    metrics = {
        "nn": k,
        "ARI": adjusted_rand_score(y_val_num, predicted_num),
        "NMI": normalized_mutual_info_score(y_val_num, predicted_num),
        "accuracy": accuracy_score(y_val_num, predicted_num),
        "macro_F1": f1_score(y_val, predicted, labels=class_names,
                                average="macro", zero_division=0),
    }

    return metrics, predicted


### kNN without PCA - k varying

In [ ]:
num_neighbors = 5
knn_neighbors = range(1, num_neighbors+1)
use_features = features_C
optimize_metric = "ARI" # ARI, NMI, accuracy, macro_F1

k_idx_mapping = {x:x-1 for x in knn_neighbors}

preprocess = Pipeline([
    ("scaler", StandardScaler()),
    ("variance", VarianceThreshold(threshold=0.0))
])

val_pred_structs = np.empty(num_neighbors, dtype=object)
metrics_df = []

X_train = preprocess.fit_transform(train[use_features])
X_val = preprocess.transform(val[use_features])
y_train = train["degradation_status"].to_numpy()
y_val = val["degradation_status"].to_numpy()

results = []

for i,k in enumerate(knn_neighbors):
    metrics, predicted = perform_knn(k, X_train, X_val, y_train, y_val)
    val_pred_structs[i] = predicted
    metrics_df.append(metrics)

metrics_df = pd.DataFrame(metrics_df).sort_values(optimize_metric, ascending=False)
print('metrics df:', metrics_df)
best_k = metrics_df["nn"].values[0]

val_predictions_best = val_pred_structs[k_idx_mapping[best_k]]

confusion_df = pd.DataFrame(
    confusion_matrix(
        predicted,
        y_val,
        labels=class_names
    ),
    index=[f"Actual {label}" for label in class_names],
    columns=[f"Predicted {label}" for label in class_names]
    )

print("\nValidation confusion matrix:")
print(confusion_df)


print("\nValidation classification report:")
print(classification_report(
    y_val,
    val_predictions_best,
    labels=class_names,
    zero_division=0
))


metrics df:    nn       ARI       NMI  accuracy  macro_F1
4   5  0.476091  0.429443  0.784639  0.588345
2   3  0.463616  0.418451  0.778516  0.584242
3   4  0.450998  0.413360  0.775354  0.582456
0   1  0.446039  0.403550  0.768590  0.582100
1   2  0.420853  0.398829  0.760066  0.576272

Validation confusion matrix:
                 Predicted Normal  Predicted Moderate  Predicted Severe  \
Actual Normal                7521                1022               269   
Actual Moderate               916                6308              1729   
Actual Severe                 201                1216              5776   
Actual Critical                 0                   1                 0   

                 Predicted Critical  
Actual Normal                     0  
Actual Moderate                   2  
Actual Severe                    25  
Actual Critical                   0  

Validation classification report:
              precision    recall  f1-score   support

      Normal       0.85   

### kNN with PCA - k varying and number of components varying

In [ ]:
num_neighbors = 5
knn_neighbors = range(1, num_neighbors+1)
use_features = features_C
optimize_metric = "ARI" # ARI, NMI, accuracy, macro_F1

k_idx_mapping = {x:x-1 for x in knn_neighbors}

preprocess = Pipeline([
    ("scaler", StandardScaler()),
    ("variance", VarianceThreshold(threshold=0.0))
])

val_pred_structs = np.empty(num_neighbors, dtype=object)
metrics_df = []


X_train_prepared = preprocess.fit_transform(train[use_features])
X_val_prepared = preprocess.transform(val[use_features])
y_train_prepared = train["degradation_status"].to_numpy()
y_val_prepared = val["degradation_status"].to_numpy()

max_components = min(X_train.shape)

# You can reduce this if the full search is slow.
pca_component_candidates = range(1, max_components + 1)

results = []

for j,c in enumerate(pca_component_candidates):

    pca = PCA(
        n_components=c,
        svd_solver="full",
        whiten=False,
        random_state=42
    )

    X_train_pca = pca.fit_transform(X_train_prepared)
    X_val_pca = pca.transform(X_val_prepared)

    explained_variance = pca.explained_variance_ratio_.sum()

    for i,k in enumerate(knn_neighbors):
        metrics, prediced = perform_knn(k, X_train_pca, X_val_pca, y_train, y_val)
        val_pred_structs[i] = prediced
        metrics_df.append(metrics)

metrics_df = pd.DataFrame(metrics_df).sort_values(optimize_metric, ascending=False)
print('metrics df:', metrics_df)
best_k = metrics_df["nn"].values[0]

val_predictions_best = val_pred_structs[k_idx_mapping[best_k]]

confusion_df = pd.DataFrame(
    confusion_matrix(
        val_predictions_best,
        y_val,
        labels=class_names
    ),
    index=[f"Actual {label}" for label in class_names],
    columns=[f"Predicted {label}" for label in class_names]
    )

print("\nValidation confusion matrix:")
print(confusion_df)


print("\nValidation classification report:")
print(classification_report(
    y_val,
    val_predictions_best,
    labels=class_names,
    zero_division=0
))


metrics df:     nn       ARI       NMI  accuracy  macro_F1
19   5  0.476091  0.429443  0.784639  0.588345
17   3  0.463616  0.418451  0.778516  0.584242
18   4  0.450998  0.413360  0.775354  0.582456
15   1  0.446039  0.403550  0.768590  0.582100
16   2  0.420853  0.398829  0.760066  0.576272
14   5  0.313943  0.300427  0.678540  0.507620
12   3  0.306661  0.295450  0.673857  0.505111
13   4  0.291290  0.287005  0.667454  0.499650
9    5  0.291077  0.287539  0.649684  0.482477
4    5  0.281616  0.286335  0.634155  0.469435
10   1  0.281582  0.264792  0.657008  0.497122
7    3  0.269112  0.266126  0.635076  0.473353
2    3  0.257186  0.259108  0.619827  0.461008
11   2  0.255522  0.271197  0.650284  0.490841
8    4  0.251459  0.256919  0.622308  0.460622
3    4  0.248353  0.257166  0.606940  0.445725
5    1  0.227876  0.218454  0.602898  0.449704
0    1  0.198262  0.190285  0.572921  0.426612
6    2  0.195984  0.226239  0.595533  0.439156
1    2  0.170861  0.206767  0.565917  0.413080



##### Rand Index (RI)

This is a statistic to measure the similarity between two clusterings. Its formula is
$RI=\frac{TP+TN}{TP+TN+FP+FN}$
where $TP,TN,FP,FN$ are the true positives, true negatives, false positives and false negatives, respectively.
It is computed by looking at all the possible $\binom{n}{2}$ pairs in the dataset and comparing the true label with the clustered (predicted) label.

##### ARI (Adjusted Rand Score)

This is a statistic to measure the quality of data clustering. It measures the accuracy of the predicted values with respect to the ground-truth values. It is a normalized version of the Rand Index and is less sensible to noise and chance. Its formula is
$ARI = \frac{RI - E[RI]}{\text{max}(RI)-E[RI]}$

What are max(RI) and E[RI] and how are they calculated?

We first make a contingency table, ie. a square matrix where position $(i,j)$ corresponds with the number of items in the dataset that are both in the ground-truth cluster $i$ and in the predicted cluster $j$. Next we define $a_i$ to be the sum of row $i$, ie. the total number of items in ground-truth cluster $i$, and $b_j$ to be the sum of column $j$, ie. the total number of items in the predicted cluster $j$. we can then calculate $\text{max}(RI)$ and $E[RI]$:

$\text{max}(RI)=\frac{1}{2}(\sum_i\binom{a_i}{2}+\sum_j\binom{b_j}{2})$
$E[RI]=\binom{n}{2}^{-1}(\sum_i\binom{a_i}{2}+\sum_j\binom{b_j}{2})$

$E[RI]$ represents the score one would get if one randomly shuffled the cluster labels around while keeping the cluster sizes the same, while $\text{max}(RI)$ is the average of the internal pairs in both groupings, which represents the maximum possible value of the agreement between ground-truth and predicted labeling.

##### NMI (Normalized Mutual Information)

This is another accuracy measure, but unlike ARI it doesn't count pairings, but rather uses Shannon-entropy and represents the amount of information one cluster shares with another. It quantifies the reduction in uncertainty about the true cluster labels given knowledge of the predicted cluster labels.
$NMI(U,V)=\frac{\text{MI}(U,V)}{\text{Normalizer}(H(U),H(V))}$

where $H(U),H(V)$ are the entropies of the datasets $U$ and $V$ respectively.

$\text{MI}(U,V)$ is the Mutual Information between $U$ and $V$ and is calculated as
$\text{MI}(U,V)=\sum_i\sum_j P(i,j)\text{log}(\frac{P(i,j)}{P(i)P(j)})$

This determines how much knowing clustering $V$ reduces our uncertainty about clustering $U$. one can see that if $U$ and $V$ are completely independent, then $P(i,j)=P(i)P(j)$ and $\text{MI}(U,V)=0$. If $U=V$ then $\text{MI}(U,V)$ equals the total entropy of the common dataset. Here, $P(i)$ is the probability of a random point landing in cluster $i$ and $P(i,j)$ is the probability of a random point landing in true cluster $i$ and predicted cluster $j$.

$\text{Normalizer}(H(U),H(V))$ is a normalizer for the NMI quantity. We use this because the mutual information is uncapped, and we wish to normalize this to ensure the NMI is between 0 and 1. In scikit-learn, this is usually calculated as $\text{Normalizer}(H(U),H(V))=\frac{1}{2}(H(U)+U(V))$ (ie. the arithmetic mean), but one might also use the harmonic mean: $\sqrt{H(U)\cdot H(V)}$.

##### Accuracy

This is simply calculated as $\frac{\text{number of correct predictions}}{\text{total number of predictions}}$. It is a simple measure and can be misleading when the sizes of the clusters are not balanced.

##### Macro F1

This calculates $\frac{1}{n}\sum_{i=1}^n\text{F1}_{\text{class}\ i}$, where $\text{F1}_{\text{class}\ i}=2\cdot\frac{\text{precision}_i\cdot\text{recall}_i}{\text{precision}_i+\text{recall}_i}$, with $\text{recall}_i=\frac{TP_i}{TP_i+FN_i}$ and $\text{precision}_i=\frac{TP_i}{TP_i+FP_i}$.